# Paper Citation Trend — yearly incoming citations per publication

For every publication, its **year-by-year incoming citations**, anchored at the publication year:
- `p2p` — **paper→paper** citations per year, from the Dimensions reference graph (kernel as in
  `OpenAlex/notebook/paper_citation_trend.ipynb`);
- `pat2p` — **patent→paper** citations per year, from the `publication_ids` of the Dimensions
  `patents` folder, dated by the patent's `granted_year` (else `publication_year`); `pat2p_us` is
  the US-jurisdiction subset.

Dimensions does not tag a patent's references as examiner / applicant, so the OpenAlex
`pat2p_examiner` / `pat2p_non_examiner` split has no twin here; the total and the US subset are
what can be said. The patent linkage is Dimensions' own (all jurisdictions, granted and
applications), not PCS, so `pat2p` and the OpenAlex `pat2p_*` are different populations.

## Input
```
Dimensions/cache/paper_graph.npz              # c_from, c_to, year, uni_mag
<dump>/patents/patents_*                      # id, jurisdiction, granted_year, publication_year, filing_status, publication_ids[]
Dimensions/cache/patent2pub_edges.parquet     # built here: one row per (patent, cited publication), de-duplicated
```

## Output
`Dimensions/output/paper_citation_trend.parquet` — `paper_id, pub_year, cite_year, yrs_since_pub, p2p, pat2p, pat2p_us`
(one row per publication × citing year with ≥ 1 citation of any channel).

In [1]:
import os, sys, gc, glob, time
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/Dimensions')
import dim_common as dim
ROOT = dim.BASE; OUT = dim.OUT
print('dump:', dim.ROOT)
import duckdb
OUT_FP  = f'{OUT}/paper_citation_trend.parquet'
RUN_PAT2P = True        # set False for the paper->paper half only
# Everything below is fed by notebook/references_w_year.ipynb: it writes the per-publication
# map (year + source), the scalar and author parts, and the edge table with both years and both
# source ids. Build it once before running this notebook.
assert dim.have_consolidated(), (
    'run notebook/references_w_year.ipynb first -- it builds the map, the scalar parts and the edge table')
dim.summary()

dump: /project/jevans/dimensions/dimensions/dimensions_june_2025
dump     : /project/jevans/dimensions/dimensions/dimensions_june_2025
cache    : /project/jevans/Dawoon/Science of Science/Dimensions/cache
output   : /project/jevans/Dawoon/Science of Science/Dimensions/output
  consolidated edge table: present
  map            2.80 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/pub_year_source_map.npz
  graph         19.00 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_graph.npz
  csr           21.49 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_csr.npz
  journal        1.24 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_journal.parquet
  fos            0.77 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_fos.parquet
  pat2pub      not built  /project/jevans/Dawoon/Science of Science/Dimensions/cache/patent2pub_edges.parquet
  scalars       4219 parts  /project/jevans/Dawoon/Science o

## 1. paper→paper (p2p) yearly — vectorised over the citation graph

In [2]:
%%time
c_from, c_to, year, uni_mag = dim.load_graph()
n = len(uni_mag); Y0 = 1700
cy = year[c_from].astype(np.int32); del c_from; gc.collect()   # citing year
diff = cy - year[c_to]
m = diff >= 0; del diff; gc.collect()
cited = c_to[m].astype(np.int64); del c_to; gc.collect()
cyv = cy[m].astype(np.int64); del cy, m; gc.collect()
key = cited * 400 + (cyv - Y0); del cited, cyv; gc.collect()
u, cnt = np.unique(key, return_counts=True); del key; gc.collect()
p2p = pd.DataFrame({'code': (u // 400).astype(np.int64), 'cite_year': (u % 400 + Y0).astype(np.int32),
                    'p2p': cnt.astype(np.int64)}); del u, cnt; gc.collect()
print(f'p2p (paper, year) rows: {len(p2p):,}')

graph cache present: /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_graph.npz
p2p (paper, year) rows: 573,297,653


## 2. patent→paper (pat2p) yearly — from the patents folder

In [3]:
%%time
# 2. patent -> paper, per (publication, patent year); all jurisdictions and the US subset
PFILES = dim.patent_files()
con = duckdb.connect()
con.execute("SET memory_limit='120GB'"); con.execute(f"SET temp_directory='{dim.CACHE}/duckdb_tmp'")
con.execute('SET preserve_insertion_order=false')
if RUN_PAT2P:
    if not os.path.exists(dim.PAT2PUB):
        t0 = time.time()
        con.execute(f"""
        COPY (
          SELECT id AS patent_id, jurisdiction, filing_status,
                 coalesce(TRY_CAST(granted_year AS INTEGER), TRY_CAST(publication_year AS INTEGER)) AS cite_year,
                 unnest(list_distinct(publication_ids)) AS paper_id
          FROM read_parquet({PFILES!r}, union_by_name=true)
          WHERE publication_ids IS NOT NULL AND len(publication_ids) > 0
        ) TO '{dim.PAT2PUB}.tmp' (FORMAT PARQUET, COMPRESSION ZSTD)""")
        os.replace(dim.PAT2PUB + '.tmp', dim.PAT2PUB)
        print(f'[{time.time()-t0:.0f}s] patent->publication edges -> {dim.PAT2PUB}')
    st = con.execute(f"""SELECT count(*) AS edges, count(DISTINCT patent_id) AS patents, count(DISTINCT paper_id) AS papers,
                          count(*) FILTER (WHERE jurisdiction = 'US') AS us_edges, count(*) FILTER (WHERE cite_year IS NULL) AS undated,
                          min(cite_year) AS y0, max(cite_year) AS y1 FROM read_parquet('{dim.PAT2PUB}')""").df().iloc[0]
    print('  ' + '  '.join(f'{k} {int(v):,}' for k, v in st.items()))
    pat2p = con.execute(f"""
        SELECT TRY_CAST(substr(paper_id, 5) AS BIGINT) AS oaid, cite_year::INTEGER AS cite_year,
               count(*)::BIGINT AS pat2p, count(*) FILTER (WHERE jurisdiction = 'US')::BIGINT AS pat2p_us
        FROM read_parquet('{dim.PAT2PUB}') WHERE cite_year IS NOT NULL AND paper_id LIKE 'pub.%'
        GROUP BY 1, 2""").df()
    print(f'pat2p (publication, year) rows: {len(pat2p):,}')
else:
    pat2p = pd.DataFrame({'oaid': pd.Series(dtype='int64'), 'cite_year': pd.Series(dtype='int32'),
                          'pat2p': pd.Series(dtype='int64'), 'pat2p_us': pd.Series(dtype='int64')})
con.close()

[98s] patent->publication edges -> /project/jevans/Dawoon/Science of Science/Dimensions/cache/patent2pub_edges.parquet
  edges 27,889,634  patents 4,467,532  papers 5,273,583  us_edges 21,940,598  undated 0  y0 1,954  y1 2,025
pat2p (publication, year) rows: 14,773,094


## 3. Assemble + save

In [4]:
%%time
# 3. Assemble + save. p2p code -> accession; pat2p uses the accession directly.
p2p['oaid'] = uni_mag[p2p['code'].to_numpy()]
pub = pd.DataFrame({'oaid': uni_mag, 'pub_year': year})
tr = p2p[['oaid', 'cite_year', 'p2p']].merge(pat2p, on=['oaid', 'cite_year'], how='outer')
tr = tr.merge(pub, on='oaid', how='left')
for c in ['p2p', 'pat2p', 'pat2p_us']:
    tr[c] = tr[c].fillna(0).astype('int64')
tr = tr.dropna(subset=['pub_year'])
tr['cite_year'] = tr['cite_year'].astype(int); tr['pub_year'] = tr['pub_year'].astype(int)
tr = tr[tr['cite_year'] >= tr['pub_year']]
tr['yrs_since_pub'] = tr['cite_year'] - tr['pub_year']
del p2p, pat2p, pub; gc.collect()
tr['paper_id'] = dim.ID_PREFIX + tr['oaid'].astype('int64').astype(str)
tr = tr[['paper_id', 'pub_year', 'cite_year', 'yrs_since_pub', 'p2p', 'pat2p', 'pat2p_us']]
tr.to_parquet(OUT_FP, index=False)
print(f'WROTE {OUT_FP}  ({len(tr):,} rows)')
print(f'  publications covered: {tr.paper_id.nunique():,} | mean p2p/row {tr.p2p.mean():.2f} | rows with pat2p>0: {(tr.pat2p>0).sum():,}')
display(tr.head(8))

WROTE /project/jevans/Dawoon/Science of Science/Dimensions/output/paper_citation_trend.parquet  (577,326,798 rows)
  publications covered: 84,080,875 | mean p2p/row 3.70 | rows with pat2p>0: 14,747,050


,paper_id,pub_year,cite_year,yrs_since_pub,p2p,pat2p,pat2p_us
0,pub.1000000002,1999,2000,1,6,0,0
1,pub.1000000002,1999,2001,2,4,0,0
2,pub.1000000002,1999,2002,3,4,0,0
3,pub.1000000002,1999,2003,4,4,0,0
4,pub.1000000002,1999,2004,5,3,0,0
5,pub.1000000002,1999,2005,6,1,0,0
6,pub.1000000002,1999,2006,7,3,0,0
7,pub.1000000002,1999,2007,8,6,0,0
